# Baseline 5: Unrelated-questions classifier
This notebook implements the black box method **"unrelated questions classifier "** from Pacchiardi et al., [How to Catch an AI Liar](https://arxiv.org/abs/2309.15840).

After the model produces a response, we append one of 39 fixed, *unrelated* yes/no follow-up questions (e.g. "Have you ever lied? Answer 'yes' or 'no' and nothing else.") to the same conversation, sample a response, and parse it as yes (1) / no (0) /unparseable. 

Doing this for all 39 questions gives a 39-dimensional binary responses of the model's "state" after a deceptive vs. honest answer. We then use a simple logistic regression classifier on this vector as the lie detector.

## 0️⃣ Setup

In [ ]:
import json

import numpy as np
import pandas as pd

## 1️⃣ Load the precomputed fingerprints

In [ ]:
train_df = pd.read_csv("data/pacchiardi_gemma_training.csv")
with open("data/pacchiardi_questions.json") as f:
    QUESTIONS = json.load(f)

print(f"{len(train_df)} examples across {train_df['dataset'].nunique()} datasets")
print(f"{len(QUESTIONS)} elicitation questions")
train_df.head()

## 2️⃣ Build the training matrix

In [ ]:
import ast

X_lie = np.array([ast.literal_eval(s) for s in train_df["lie_binary_features"]], dtype=np.float64)
X_truth = np.array([ast.literal_eval(s) for s in train_df["truth_binary_features"]], dtype=np.float64)
dataset_lie = train_df["dataset"].values

X = np.vstack([X_lie, X_truth])
y = np.concatenate([np.ones(len(X_lie)), np.zeros(len(X_truth))])
datasets = np.concatenate([dataset_lie, dataset_lie])   # which source dataset each row came from

assert X.shape[1] == len(QUESTIONS)
print(f"X: {X.shape} | y: {y.shape} | lie={int(y.sum())} honest={int((1 - y).sum())}")

## 3️⃣ Fit and evaluate

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

clf = LogisticRegression(max_iter=2000, random_state=42).fit(X_tr, y_tr)
p_te = clf.predict_proba(X_te)[:, 1]

print(f"holdout AUROC:   {roc_auc_score(y_te, p_te):.3f}")
print(f"holdout bal-acc: {balanced_accuracy_score(y_te, clf.predict(X_te)):.3f}")

In [ ]:
# Save the trained classifier for submission
np.savez(
    "unrelated_questions_probe.npz",
    coef=clf.coef_[0].astype(np.float32),
    intercept=np.array(float(clf.intercept_[0]), dtype=np.float32),
    questions=np.array(QUESTIONS),
    model_id=np.array("google/gemma-3-27b-it"),
    temperature=np.array(0.7, dtype=np.float32),
    max_new_tokens=np.array(6),
    yes_tokens=np.array(["yes", "absolutely", "certainly", "undoubtedly"]),
    no_tokens=np.array(["no", "never"]),
)
print(f"saved unrelated_questions_probe.npz | {len(QUESTIONS)} questions | dim {clf.coef_[0].shape[0]}")